# PromptTemplate — 프롬프트를 재사용 가능한 틀로 만들기

Ch01에서 LLM 호출을 배웠다면, 이제 프롬프트를 체계적으로 관리하는 법을 배워요.

매번 프롬프트를 하드코딩하는 대신, 변수 슬롯이 있는 **템플릿(template)**을 만들어두면 같은 구조의 프롬프트를 여러 상황에서 재사용할 수 있어요.

## 학습 목표

- `PromptTemplate`으로 변수를 포함한 재사용 가능한 프롬프트를 만들 수 있어요
- `ChatPromptTemplate`으로 System / Human / AI 역할을 구분한 대화형 프롬프트를 구성할 수 있어요
- `partial()`과 동적 함수 변수로 프롬프트를 더 유연하게 다룰 수 있어요

## 사전 지식

- Ch01의 `ChatOpenAI` 기본 호출 방법
- LCEL 파이프라인 (`|` 연산자) 기초

---

> **PromptTemplate 동작 흐름** — 아래 다이어그램은 변수가 어떻게 최종 프롬프트로 조립되어 LLM에 전달되는지 보여줘요.

```mermaid
flowchart LR
    V1["변수 role<br/>'파이썬 전문가'"]:::input
    V2["변수 topic<br/>'데코레이터'"]:::input
    T["PromptTemplate<br/>템플릿 문자열<br/>'당신은 {role}입니다.<br/>{topic}에 대해 설명해주세요.'"]:::process
    P["완성된 프롬프트<br/>'당신은 파이썬 전문가입니다.<br/>데코레이터에 대해 설명해주세요.'"]:::output
    LLM["LLM<br/>ChatOpenAI"]:::external
    R["응답"]:::output

    V1 --> T
    V2 --> T
    T -->|"format() / invoke()"| P
    P --> LLM
    LLM --> R

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef error fill:#f8d7da,stroke:#dc3545,color:#721c24
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef external fill:#d1ecf1,stroke:#17a2b8,color:#0c5460
```

## 1. PromptTemplate

`PromptTemplate`은 문자열 기반의 간단한 프롬프트를 생성하는 템플릿입니다.

### 주요 특징:
- 중괄호 `{}`를 사용하여 변수 정의
- `from_template()` 메서드로 간편하게 생성
- `format()` 또는 `invoke()`로 변수에 값을 채워 최종 프롬프트 생성

> 🔑 **핵심 개념**: `PromptTemplate`은 프롬프트의 "틀"이에요. `{변수명}` 플레이스홀더 덕분에 같은 구조의 프롬프트를 수백 가지 입력에 재사용할 수 있어요. 하드코딩된 프롬프트와의 가장 큰 차이는 **재사용성**과 **관심사 분리**입니다.

> 🎯 **강의 포인트**: `format()`은 완성된 문자열을 반환하고, `invoke()`는 LangChain 메시지 객체를 반환해요. 체인에서 사용할 때는 `invoke()`를 쓰지만, 프롬프트가 어떻게 생성됐는지 확인하려면 `format()`이 편리해요. 수업에서 `format()`으로 먼저 결과를 출력한 뒤 체인으로 연결하면 이해가 빠릅니다.

In [ ]:
# 필수 라이브러리 import
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

# 모델 초기화
# temperature=0: 일관된 출력을 위해 설정
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

### 1.1 기본 사용법


In [2]:
# 1단계: 템플릿 정의 - 중괄호 {}로 변수 지정
template = "당신은 {role}입니다. {topic}에 대해 설명해주세요."

# 2단계: PromptTemplate 생성
prompt = PromptTemplate.from_template(template)

# 3단계: format() 메서드로 변수에 값을 채워 최종 프롬프트 생성
result = prompt.format(role="파이썬 전문가", topic="데코레이터")

print("생성된 프롬프트:")
print(result)


생성된 프롬프트:
당신은 파이썬 전문가입니다. 데코레이터에 대해 설명해주세요.


### 1.2 체인과 함께 사용하기

`PromptTemplate`을 LLM과 연결하여 체인을 구성할 수 있습니다.


In [3]:
# 1단계: 템플릿 정의
prompt = PromptTemplate.from_template(
    "당신은 {role}입니다. {topic}에 대해 {length} 단어 이내로 설명해주세요."
)

# 2단계: 체인 구성 (프롬프트 -> 모델 -> 파서)
chain = (
    prompt
    | model
    | StrOutputParser()
)

# 3단계: 체인 실행
result = chain.invoke({
    "role": "데이터 과학자",
    "topic": "머신러닝",
    "length": "50"
})

print("=" * 60)
print("LLM 응답:")
print("=" * 60)
print(result)


LLM 응답:
머신러닝은 데이터에서 패턴을 학습하여 예측하거나 결정을 내리는 알고리즘과 기술의 집합입니다. 통계, 컴퓨터 과학 및 인공지능을 결합하여, 모델은 주어진 데이터로부터 자동으로 개선되고 최적화됩니다. 다양한 애플리케이션에서 활용됩니다.


## 2. ChatPromptTemplate — 역할을 구분한 대화형 프롬프트

`PromptTemplate`이 단순 문자열이라면, `ChatPromptTemplate`은 **역할(role)** 단위로 메시지를 구성해요.
LLM은 각 역할을 구분해서 읽기 때문에, 동일한 내용이라도 역할 배치에 따라 응답 품질이 달라져요.

아래 구조도는 세 가지 역할이 어떻게 조합되어 LLM에 전달되는지 보여줘요.

```mermaid
flowchart TD
    SV["변수 role<br/>'파이썬 프로그래밍 강사'"]:::input
    HV["변수 question<br/>'리스트와 튜플의 차이점은?'"]:::input

    SM["SystemMessage<br/>'당신은 {role}입니다.<br/>항상 친절하고 명확하게 답변합니다.'"]:::process
    HM["HumanMessage<br/>'{question}'"]:::process

    CP["ChatPromptTemplate<br/>메시지 배열 조립"]:::process

    LLM["LLM<br/>ChatOpenAI"]:::external
    R["AIMessage<br/>응답"]:::output

    SV --> SM
    HV --> HM
    SM --> CP
    HM --> CP
    CP --> LLM
    LLM --> R

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef error fill:#f8d7da,stroke:#dc3545,color:#721c24
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef external fill:#d1ecf1,stroke:#17a2b8,color:#0c5460
```

### 주요 역할(role) 종류

| 역할 | 튜플 키 | 역할 설명 |
|------|---------|-----------|
| System | `"system"` | LLM의 행동 방식·페르소나 정의 |
| Human | `"human"` | 사용자 질문·요청 |
| AI | `"ai"` | LLM의 이전 응답 (Few-Shot에 활용) |

> 🔑 **핵심 개념**: `PromptTemplate`은 단순 텍스트 완성, `ChatPromptTemplate`은 대화형 메시지 조립이에요. 현업에서 LLM을 쓰는 대부분의 경우에는 `ChatPromptTemplate`을 사용해요. System / Human 역할 구분이 응답 품질을 결정하는 핵심이기 때문이에요.

> 🎯 **강의 포인트**: `("system", "당신은 {role}입니다.")`처럼 튜플 형식을 쓰는 게 `SystemMessagePromptTemplate.from_template(...)`과 동일하지만 훨씬 간결해요. 실무에서는 튜플 형식이 표준처럼 쓰입니다. 두 방식이 동일하다는 것을 코드로 보여주면 혼란이 줄어요.

### 2.1 기본 사용법 - 시스템과 사용자 메시지


In [4]:
# 1단계: ChatPromptTemplate 정의
#        - system: LLM의 역할과 행동 방식 정의
#        - human: 사용자의 질문
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 {role}입니다. 항상 친절하고 명확하게 답변합니다."),
    ("human", "{question}")
])

# 2단계: 체인 구성
chain = (
    chat_prompt
    | model
    | StrOutputParser()
)

# 3단계: 실행
result = chain.invoke({
    "role": "파이썬 프로그래밍 강사",
    "question": "리스트와 튜플의 차이점을 설명해주세요."
})

print("=" * 60)
print("LLM 응답:")
print("=" * 60)
print(result)


LLM 응답:
리스트(List)와 튜플(Tuple)은 파이썬에서 데이터를 저장하는데 사용되는 기본 자료형입니다. 두 자료형 모두 순서가 있는 컬렉션이지만, 몇 가지 중요한 차이점이 있습니다.

1. **변경 가능성 (Mutability)**:
   - **리스트**: 변경 가능합니다. 즉, 리스트 내의 요소를 추가, 삭제 또는 변경할 수 있습니다.
     ```python
     my_list = [1, 2, 3]
     my_list[0] = 10  # 리스트의 첫 번째 요소를 변경
     my_list.append(4)  # 리스트에 새로운 요소 추가
     print(my_list)  # 출력: [10, 2, 3, 4]
     ```
     
   - **튜플**: 변경 불가능합니다. 튜플을 한 번 정의하면 그 내용을 수정할 수 없습니다.
     ```python
     my_tuple = (1, 2, 3)
     # my_tuple[0] = 10  # 오류: 'tuple' 객체는 변경할 수 없습니다.
     ```

2. **구조**:
   - **리스트**: 대괄호 `[]`로 정의합니다.
     ```python
     my_list = [1, 2, 3]
     ```
   - **튜플**: 괄호 `()`로 정의합니다.
     ```python
     my_tuple = (1, 2, 3)
     ```

3. **성능**:
   - 튜플은 리스트보다 메모리 효율성이 더 높습니다. 변경이 불가능하므로, 파이썬에서 튜플을 처리하는 속도가 더 빠를 수 있습니다. 

4. **사용 용도**:
   - 리스트는 데이터의 순서가 중요하고, 자주 변경될 가능성이 있는 경우에 사용합니다.
   - 튜플은 데이터가 변경되지 않아야 하거나, 고정된 데이터 시퀀스가 필요한 경우에 사용합니다. 예를 들어, 함수의 반환값으로 여러 값을 한 번에 전달할 때 튜플을 사용할 수 있습니다.

5. **내장 메서드**:
   - 리스트는 다양한 메서드(

### 2.2 대화 이력 포함하기 - Few-Shot 예제

AI의 이전 응답을 포함하여 대화의 맥락을 제공할 수 있습니다. 이는 Few-Shot Learning에 유용합니다.


In [5]:
# 1단계: 대화 이력을 포함한 ChatPromptTemplate 정의
#        - system: 역할 정의
#        - human/ai: 예시 대화 (Few-Shot 예제)
#        - human: 실제 질문
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 감정 분석 전문가입니다. 문장의 감정을 '긍정', '부정', '중립'으로 분류합니다."),
    ("human", "이 영화 정말 재미있었어요!"),
    ("ai", "감정: 긍정"),
    ("human", "배송이 너무 느려서 실망했습니다."),
    ("ai", "감정: 부정"),
    ("human", "{text}")
])

# 2단계: 체인 구성
chain = (
    chat_prompt
    | model
    | StrOutputParser()
)

# 3단계: 실행
result = chain.invoke({
    "text": "가격 대비 괜찮은 것 같아요."
})

print("=" * 60)
print("감정 분석 결과:")
print("=" * 60)
print(result)


감정 분석 결과:
감정: 긍정


### 2.3 복잡한 프롬프트 - 실용 예제

실제 업무에서 사용할 수 있는 구조화된 프롬프트 예제입니다.


In [ ]:
# ---------------------------------------------------
# 복잡한 ChatPromptTemplate 활용 - 구조화된 리뷰 분석
# ---------------------------------------------------

# 시나리오: 고객 리뷰를 분석하여 요약 보고서를 작성하는 시스템

# 1단계: 복잡한 ChatPromptTemplate 정의
#        - system: 분석 전문가 역할 + 출력 형식 지정
#        - human: 제품명, 카테고리, 리뷰 텍스트를 변수로 전달
review_prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 고객 서비스 분석 전문가입니다.
다음 형식으로 리뷰를 분석해주세요:

1. 전체 평가 (1-5점)
2. 주요 긍정 포인트
3. 주요 부정 포인트
4. 개선 제안사항

간결하고 명확하게 작성해주세요."""),
    ("human", """제품: {product_name}
카테고리: {category}

고객 리뷰:
{review_text}""")
])

# 2단계: 체인 구성
chain = (
    review_prompt
    | model
    | StrOutputParser()
)

# 3단계: 실행
result = chain.invoke({
    "product_name": "무선 이어폰 Pro X",
    "category": "전자제품",
    "review_text": """
음질은 정말 훌륭하고 노이즈 캔슬링 기능이 뛰어납니다.
하지만 배터리 지속 시간이 광고보다 짧고, 케이스가 너무 커서 휴대하기 불편합니다.
가격 대비로는 만족스럽지만, 앱 연결이 가끔 끊기는 문제가 있어요.
    """
})

print("=" * 60)
print("리뷰 분석 결과:")
print("=" * 60)
print(result)

## 3. 템플릿 변수 활용법 — partial()과 동적 변수

> 💡 **실무 팁:** 같은 템플릿을 여러 팀에서 쓴다면 `partial()`로 공통 변수를 미리 고정해두면 중복 코드를 없앨 수 있어요.

프롬프트 템플릿에서 변수를 효과적으로 사용하는 다양한 방법을 살펴볼게요.

> 🎯 **강의 포인트**: `partial()`은 함수형 프로그래밍의 부분 적용(Partial Application) 개념이에요. 공통 설정(회사명, 언어 설정, 출력 형식)을 한 번 고정하고 나머지 변수만 채우면, 팀 내 모든 체인이 일관된 프롬프트 구조를 공유할 수 있습니다. 설정 변경 시 한 곳만 수정하면 되는 유지보수 이점도 있어요.

### 3.1 부분 변수 채우기 - partial()

일부 변수를 미리 고정하고, 나중에 나머지 변수만 채울 수 있습니다.


In [ ]:
# ---------------------------------------------------
# partial() - 자주 쓰는 변수를 미리 고정하여 재사용
# ---------------------------------------------------

# 시나리오: 특정 회사의 여러 부서에서 같은 템플릿을 사용하지만,
#           회사명은 항상 동일하게 유지

# 1단계: 템플릿 정의
template = """회사: {company_name}
부서: {department}
직원: {employee_name}

위 정보로 환영 메시지를 작성해주세요."""

prompt = PromptTemplate.from_template(template)

# 2단계: 회사명을 미리 고정 (partial)
# partial(): 자주 변하지 않는 변수를 미리 채워 재사용 가능한 프롬프트 생성
partial_prompt = prompt.partial(company_name="테크노바 주식회사")

# 3단계: 나머지 변수만 채워서 사용
#        - partial_prompt는 이미 company_name이 고정된 상태
#        - department와 employee_name만 바꿔서 여러 부서에 재사용
message1 = partial_prompt.format(
    department="개발팀",
    employee_name="김철수"
)

message2 = partial_prompt.format(
    department="마케팅팀",
    employee_name="이영희"
)

print("=" * 60)
print("메시지 1:")
print("=" * 60)
print(message1)
print()
print("=" * 60)
print("메시지 2:")
print("=" * 60)
print(message2)

### 3.2 동적 변수 — 함수로 자동 생성하기

> **주의:** `partial(current_time=get_current_time)`처럼 함수를 **호출하지 않고** 전달해야 해요. `get_current_time()`처럼 괄호를 붙이면 값이 고정되어버려요.

변수 값을 함수로 동적으로 생성할 수 있어요. 시간·날짜·계산 값 등을 자동으로 채울 때 유용합니다.

In [8]:
from datetime import datetime

# 시나리오: 일일 보고서를 자동으로 생성하는 시스템
#           현재 시간을 자동으로 포함

# 1단계: 현재 시간을 반환하는 함수 정의
def get_current_time():
    """현재 시간을 한국어 형식으로 반환"""
    return datetime.now().strftime("%Y년 %m월 %d일 %H시 %M분")

# 2단계: 템플릿 정의 및 함수를 partial로 연결
prompt = PromptTemplate.from_template(
    """작성 시간: {current_time}
보고자: {reporter}
내용: {content}

위 정보로 일일 보고서를 작성해주세요."""
)

# partial_variables에 함수를 전달하면 실행 시점에 함수가 호출됨
prompt_with_time = prompt.partial(current_time=get_current_time)

# 3단계: 체인 구성
chain = (
    prompt_with_time
    | model
    | StrOutputParser()
)

# 4단계: 실행 - 실행할 때마다 현재 시간이 자동으로 채워짐
result = chain.invoke({
    "reporter": "김개발",
    "content": "API 성능 최적화 작업 완료. 응답 시간 30% 개선됨."
})

print("=" * 60)
print("생성된 일일 보고서:")
print("=" * 60)
print(result)


생성된 일일 보고서:
### 일일 보고서

**작성 시간:** 2025년 11월 24일 09시 54분  
**보고자:** 김개발  

---

**주요 사항:**

- **작업 내용:** API 성능 최적화 작업 완료
- **성과:** 응답 시간 30% 개선

---

**상세 내용:**

본 보고서는 2025년 11월 24일에 완료된 API 성능 최적화 작업에 대한 내용입니다. 이번 작업을 통해 API의 응답 시간이 30% 개선되어, 사용자 경험이 크게 향상되었습니다.

**향후 계획:**

- 지속적인 모니터링을 통해 추가적인 성능 개선 가능성을 검토할 예정입니다.
- 사용자 피드백을 수집하여 기능 개선 및 추가 요청 사항을 반영할 계획입니다.

---

**기타:**  
API 성능 향상으로 인한 시스템 안정성 및 신뢰성 증가를 기대하며, 앞으로도 성능 최적화를 지속적으로 추진할 계획입니다.

감사합니다.


### 3.3 템플릿 조합하기

여러 템플릿을 조합하여 복잡한 프롬프트를 구성할 수 있습니다.


In [ ]:
from langchain_core.prompts import SystemMessagePromptTemplate, HumanMessagePromptTemplate

# ---------------------------------------------------
# 템플릿 조합 - 역할별로 따로 정의하고 나중에 합치기
# ---------------------------------------------------

# 시나리오: 시스템 메시지와 사용자 메시지를 별도로 관리하고 조합

# 1단계: 각 역할별로 템플릿 정의
# SystemMessagePromptTemplate: LLM의 역할과 응답 스타일 정의
system_template = SystemMessagePromptTemplate.from_template(
    "당신은 {role}입니다. {style} 스타일로 답변해주세요."
)

# HumanMessagePromptTemplate: 사용자 질문 템플릿
human_template = HumanMessagePromptTemplate.from_template(
    "{question}"
)

# 2단계: 템플릿 조합
# from_messages()에 객체 직접 전달 가능 (튜플 문자열 대신)
chat_prompt = ChatPromptTemplate.from_messages([
    system_template,
    human_template
])

# 3단계: 체인 구성
chain = (
    chat_prompt
    | model
    | StrOutputParser()
)

# 4단계: 실행
result = chain.invoke({
    "role": "유머러스한 역사 선생님",
    "style": "재미있고 친근한",
    "question": "조선시대 한글 창제의 의의를 설명해주세요."
})

print("=" * 60)
print("LLM 응답:")
print("=" * 60)
print(result)

## 4. 실전 예제: 다국어 번역 시스템

여러 개념을 종합하여 실용적인 번역 시스템을 구축해봅니다.


In [ ]:
# ---------------------------------------------------
# 실전 예제 - 다국어 번역 시스템
# ---------------------------------------------------

# 시나리오: 비즈니스 문서를 여러 언어로 번역하는 시스템
#           문맥을 고려하고 전문 용어를 정확하게 번역

# 1단계: 번역 프롬프트 템플릿 정의
#        - industry, tone: partial()로 재사용 가능하도록 변수화
#        - source_lang, target_lang, text: 호출마다 달라지는 변수
translation_prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 전문 번역가입니다.
다음 규칙을 따라주세요:
- 원문의 뉘앙스와 어조를 최대한 유지
- 전문 용어는 {industry} 분야의 표준 용어 사용
- {tone} 톤으로 번역
- 번역문만 출력 (설명 없이)"""),
    ("human", """원문 언어: {source_lang}
목표 언어: {target_lang}

원문:
{text}""")
])

# 2단계: 체인 구성
translation_chain = (
    translation_prompt
    | model
    | StrOutputParser()
)

# 3단계: 한국어 -> 영어 번역
result_en = translation_chain.invoke({
    "industry": "소프트웨어 개발",
    "tone": "공식적이고 전문적인",
    "source_lang": "한국어",
    "target_lang": "영어",
    "text": "당사의 새로운 API는 높은 처리량과 낮은 지연시간을 제공하며, 마이크로서비스 아키텍처에 최적화되어 있습니다."
})

print("=" * 60)
print("한국어 -> 영어 번역:")
print("=" * 60)
print(result_en)
print()

# 4단계: 한국어 -> 일본어 번역
result_jp = translation_chain.invoke({
    "industry": "소프트웨어 개발",
    "tone": "공손하고 정중한",
    "source_lang": "한국어",
    "target_lang": "일본어",
    "text": "프로젝트 일정이 변경되었습니다. 새로운 마감일은 다음 주 금요일입니다."
})

print("=" * 60)
print("한국어 -> 일본어 번역:")
print("=" * 60)
print(result_jp)

## 5. 핵심 정리


In [11]:
print("💡 핵심 정리")
print("=" * 60)
print()
print("📌 PromptTemplate vs ChatPromptTemplate")
print()
print("PromptTemplate:")
print("  - 단순 텍스트 프롬프트")
print("  - 문자열 기반")
print("  - 역할 구분 없음")
print("  - 사용: 간단한 질문, 텍스트 생성")
print()
print("ChatPromptTemplate:")
print("  - 대화형 프롬프트")
print("  - 메시지 배열 기반")
print("  - system, human, ai 역할 구분")
print("  - 사용: 대화형 AI, Few-Shot 학습")
print()
print("=" * 60)
print("📌 주요 메서드")
print("=" * 60)
print("  • from_template() - 템플릿 문자열로부터 생성")
print("  • format() - 변수를 채워 문자열 반환")
print("  • invoke() - 변수를 채워 메시지 객체 반환")
print("  • partial() - 일부 변수를 미리 고정")
print()
print("=" * 60)
print("📌 언제 사용할까?")
print("=" * 60)
print()
print("✅ PromptTemplate을 사용:")
print("  - 간단한 텍스트 생성 작업")
print("  - 역할 구분이 필요 없는 경우")
print()
print("✅ ChatPromptTemplate을 사용:")
print("  - 대화형 인터페이스 구축")
print("  - 시스템/사용자 역할 명확히 구분")
print("  - Few-Shot 예제 포함")
print("  - 복잡한 프롬프트 엔지니어링")


💡 핵심 정리

📌 PromptTemplate vs ChatPromptTemplate

PromptTemplate:
  - 단순 텍스트 프롬프트
  - 문자열 기반
  - 역할 구분 없음
  - 사용: 간단한 질문, 텍스트 생성

ChatPromptTemplate:
  - 대화형 프롬프트
  - 메시지 배열 기반
  - system, human, ai 역할 구분
  - 사용: 대화형 AI, Few-Shot 학습

📌 주요 메서드
  • from_template() - 템플릿 문자열로부터 생성
  • format() - 변수를 채워 문자열 반환
  • invoke() - 변수를 채워 메시지 객체 반환
  • partial() - 일부 변수를 미리 고정

📌 언제 사용할까?

✅ PromptTemplate을 사용:
  - 간단한 텍스트 생성 작업
  - 역할 구분이 필요 없는 경우

✅ ChatPromptTemplate을 사용:
  - 대화형 인터페이스 구축
  - 시스템/사용자 역할 명확히 구분
  - Few-Shot 예제 포함
  - 복잡한 프롬프트 엔지니어링


## 6. 추가 연습 문제

직접 해보면서 학습 내용을 정리해봅시다!


### 연습 1: 이메일 자동 생성기

**과제**: 고객에게 보낼 이메일을 자동으로 생성하는 시스템을 만드세요.

**요구사항**:
- `ChatPromptTemplate` 사용
- 시스템 메시지: "당신은 고객 서비스 담당자입니다. 정중하고 친절하게 작성합니다."
- 변수: `customer_name`, `order_number`, `issue_type`, `resolution`
- 이메일 형식으로 출력

**힌트**: 아래 셀에 코드를 작성해보세요.


In [ ]:
# ============================================================
# TODO: ChatPromptTemplate으로 고객 이메일 자동 생성기를 만드세요
# 힌트: from_messages([("system", ...), ("human", ...)])로 구성하고
#       customer_name, order_number, issue_type, resolution 변수를 활용하세요
# 예상 결과: 고객명·주문번호·문의유형·해결방안이 채워진 이메일 텍스트
# ============================================================

# 여기에 코드를 작성하세요

# email_prompt = ChatPromptTemplate.from_messages([
#     ...
# ])

### 연습 1 풀이

In [ ]:
# 연습 1 풀이: 이메일 자동 생성기

# 1단계: ChatPromptTemplate 정의
email_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 고객 서비스 담당자입니다. 정중하고 친절하게 작성합니다."),
    ("human", """다음 정보로 고객에게 보낼 이메일을 작성해주세요.

고객명: {customer_name}
주문번호: {order_number}
문의 유형: {issue_type}
해결 방안: {resolution}

이메일 형식으로 작성해주세요. (인사 → 본문 → 마무리)""")
])

# 2단계: 체인 구성
email_chain = email_prompt | model | StrOutputParser()

# 3단계: 실행
result = email_chain.invoke({
    "customer_name": "김영희",
    "order_number": "ORD-2024-0315",
    "issue_type": "배송 지연",
    "resolution": "익일 특급 배송으로 재발송 예정이며, 10% 할인 쿠폰 제공"
})

print("=" * 60)
print("생성된 이메일:")
print("=" * 60)
print(result)

### 연습 2: 코드 리뷰 봇

**과제**: 코드 리뷰를 수행하는 AI 시스템을 만드세요.

**요구사항**:
- `ChatPromptTemplate` 사용
- Few-Shot 예제 2개 포함 (좋은 코드 예시, 개선이 필요한 코드 예시)
- 변수: `programming_language`, `code`
- 리뷰 형식: 장점, 개선점, 제안사항


In [ ]:
# ============================================================
# TODO: Few-Shot 예제 2개를 포함한 코드 리뷰 봇을 만드세요
# 힌트: ChatPromptTemplate.from_messages()에 ("human", ...), ("ai", ...) 쌍으로
#       예제를 넣고, 마지막에 실제 질문 ("human", "{programming_language}... {code}...")를 추가하세요
# 예상 결과: 장점 / 개선점 / 제안사항 형식의 코드 리뷰 결과
# ============================================================

# 여기에 코드를 작성하세요

# code_review_prompt = ChatPromptTemplate.from_messages([
#     ...
# ])

### 연습 2 풀이

In [ ]:
# 연습 2 풀이: 코드 리뷰 봇

# 1단계: Few-Shot 예제를 포함한 ChatPromptTemplate 정의
code_review_prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 시니어 소프트웨어 엔지니어입니다. 
코드를 리뷰할 때 다음 형식으로 답변합니다:
- 장점: 코드의 좋은 점
- 개선점: 수정이 필요한 부분
- 제안사항: 더 나은 코드를 위한 제안"""),
    # Few-Shot 예제 1: 좋은 코드
    ("human", """Python 코드를 리뷰해주세요:
```python
def calculate_average(numbers: list[float]) -> float:
    if not numbers:
        return 0.0
    return sum(numbers) / len(numbers)
```"""),
    ("ai", """- 장점: 타입 힌트를 사용하여 가독성이 좋습니다. 빈 리스트에 대한 예외 처리가 잘 되어 있습니다.
- 개선점: 특별히 큰 문제는 없습니다.
- 제안사항: docstring을 추가하면 함수의 목적을 더 명확히 전달할 수 있습니다."""),
    # Few-Shot 예제 2: 개선이 필요한 코드
    ("human", """Python 코드를 리뷰해주세요:
```python
def f(x):
    r = []
    for i in x:
        if i > 0:
            r.append(i * 2)
    return r
```"""),
    ("ai", """- 장점: 기본적인 로직은 올바르게 동작합니다.
- 개선점: 함수명 'f'와 변수명 'r', 'i'가 의미를 전달하지 못합니다. 타입 힌트가 없어 입출력 타입을 알기 어렵습니다.
- 제안사항: 리스트 컴프리헨션을 사용하면 더 파이썬다운 코드가 됩니다. 예: `return [i * 2 for i in numbers if i > 0]`"""),
    # 실제 리뷰 요청
    ("human", "{programming_language} 코드를 리뷰해주세요:\n```{programming_language}\n{code}\n```")
])

# 2단계: 체인 구성
review_chain = code_review_prompt | model | StrOutputParser()

# 3단계: 실행
result = review_chain.invoke({
    "programming_language": "Python",
    "code": """def get_user_data(id):
    import requests
    resp = requests.get(f"https://api.example.com/users/{id}")
    data = resp.json()
    return data["name"], data["email"], data["age"]"""
})

print("=" * 60)
print("코드 리뷰 결과:")
print("=" * 60)
print(result)

### 연습 3: 동적 시간 포함 알림 시스템

**과제**: 현재 시간을 자동으로 포함하는 알림 메시지 생성 시스템을 만드세요.

**요구사항**:
- `PromptTemplate` 사용
- `partial()`을 사용하여 현재 시간을 동적으로 추가
- 변수: `user_name`, `task`, `deadline`
- 함수로 현재 시간 생성


In [ ]:
# ============================================================
# TODO: partial()과 동적 함수 변수를 사용하여 현재 시간이 포함된 알림 시스템을 만드세요
# 힌트: get_current_time() 함수를 정의하고 prompt.partial(current_time=get_current_time)으로
#       함수를 전달하세요 (함수를 호출하지 않고 전달해야 실행 시점의 시간이 반영됩니다)
# 예상 결과: 알림 시간이 자동으로 채워진 업무 알림 메시지
# ============================================================

# 여기에 코드를 작성하세요

# def get_current_time():
#     ...
#
# notification_prompt = PromptTemplate.from_template(...)
# notification_prompt = notification_prompt.partial(...)

### 연습 3 풀이

In [ ]:
# 연습 3 풀이: 동적 시간 포함 알림 시스템

from datetime import datetime

# 1단계: 현재 시간을 반환하는 함수 정의
def get_current_time():
    """현재 시간을 한국어 형식으로 반환"""
    return datetime.now().strftime("%Y년 %m월 %d일 %H시 %M분")

# 2단계: 템플릿 정의
notification_prompt = PromptTemplate.from_template(
    """알림 시간: {current_time}
수신자: {user_name}

업무 내용: {task}
마감일: {deadline}

위 정보를 바탕으로 친절하고 명확한 업무 알림 메시지를 작성해주세요.
마감일까지 남은 시간을 언급하고, 격려의 말을 포함해주세요."""
)

# 3단계: partial()로 현재 시간을 동적으로 추가
notification_prompt = notification_prompt.partial(current_time=get_current_time)

# 4단계: 체인 구성 및 실행
notification_chain = notification_prompt | model | StrOutputParser()

result = notification_chain.invoke({
    "user_name": "박지민",
    "task": "분기별 매출 보고서 작성",
    "deadline": "2024년 3월 31일"
})

print("=" * 60)
print("생성된 알림 메시지:")
print("=" * 60)
print(result)

## 마무리 — 이번 노트북에서 배운 것

| 개념 | 핵심 메서드 | 언제 쓰나요? |
|------|------------|-------------|
| `PromptTemplate` | `from_template()`, `format()` | 단순 텍스트 생성, 역할 구분 불필요 시 |
| `ChatPromptTemplate` | `from_messages()`, `invoke()` | System/Human/AI 역할을 구분한 대화형 프롬프트 |
| `partial()` | `prompt.partial(key=value)` | 공통 변수를 미리 고정해 재사용 |
| 동적 함수 변수 | `partial(key=fn)` | 시간·날짜처럼 실행 시점에 계산되는 값 |

---

**다음 노트북 예고:** `02-FewShotPromptTemplate` — LLM에게 예제를 보여주며 원하는 출력 형식을 학습시키는 **퓨샷(Few-Shot) 프롬프팅**을 배워요. 단순 형식 지정보다 훨씬 세밀한 제어가 가능해집니다.